# Tema 4 · Laboratorio — Transfer learning con una CNN preentrenada

**Aprendizaje Profundo · CUNEF Universidad**

Entrenar una CNN grande desde cero necesita millones de imágenes y GPU. El **transfer learning** reaprovecha una red ya entrenada (aquí **MobileNetV2**, entrenada en ImageNet): congelamos su base y entrenamos solo una **cabeza** nueva para nuestra tarea (gatos vs. perros).

Con muy pocos datos y pocos minutos obtendrás una precisión altísima — imposible entrenando desde cero.

> **Usa GPU**: en Colab, *Entorno de ejecución → Cambiar tipo de entorno → GPU*.
> Ejecuta las celdas en orden.

## 1 · Datos: gatos vs. perros

Cargamos una parte del dataset `cats_vs_dogs` desde TensorFlow Datasets y redimensionamos a 160×160 (lo que espera MobileNetV2).

In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

tf.random.set_seed(42)
IMG = 160

# 40 % para entrenar, 10 % para validar (subconjunto pequeño a propósito)
(train_ds, val_ds), info = tfds.load(
    'cats_vs_dogs', split=['train[:40%]', 'train[40%:50%]'],
    as_supervised=True, with_info=True)

def preparar(img, label):
    img = tf.image.resize(img, (IMG, IMG))
    img = keras.applications.mobilenet_v2.preprocess_input(img)  # escala a [-1, 1]
    return img, label

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.map(preparar).shuffle(1000).batch(32).prefetch(AUTOTUNE)
val_ds = val_ds.map(preparar).batch(32).prefetch(AUTOTUNE)
print('Clases:', info.features['label'].names)

## 2 · La base preentrenada, congelada

Cargamos MobileNetV2 **sin** su capa de clasificación final (`include_top=False`) y la **congelamos**: sus pesos (que ya saben detectar bordes, texturas y formas) no se tocarán.

In [ ]:
base = keras.applications.MobileNetV2(
    input_shape=(IMG, IMG, 3), include_top=False, weights='imagenet', pooling='avg')
base.trainable = False   # congelamos la base
print('Parámetros de la base (congelados):', f'{base.count_params():,}')

## 3 · Añadir la cabeza y entrenar

Sobre la base congelada ponemos una cabeza mínima: un `Dropout` y una `Dense(1, sigmoid)` para la clasificación binaria. Solo se entrenan esos pocos parámetros.

In [ ]:
model = keras.Sequential([
    base,
    layers.Dropout(0.2),
    layers.Dense(1, activation='sigmoid'),
])
model.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

hist = model.fit(train_ds, validation_data=val_ds, epochs=3, verbose=2)

In [ ]:
val_acc = model.evaluate(val_ds, verbose=0)[1]
print(f'Accuracy en validación con transfer learning: {val_acc:.4f}')
print('¡Con solo 3 épocas y una cabeza minúscula! Eso es reaprovechar lo aprendido.')

## 4 · Ver algunas predicciones

In [ ]:
nombres = info.features['label'].names
images, labels = next(iter(val_ds))
probs = model.predict(images, verbose=0).ravel()

plt.figure(figsize=(12, 6))
for i in range(8):
    plt.subplot(2, 4, i+1)
    img = (images[i] + 1) / 2   # deshacer el preprocess para visualizar
    plt.imshow(img.numpy()); plt.axis('off')
    pred = nombres[int(probs[i] > 0.5)]
    real = nombres[int(labels[i])]
    ok = '✓' if pred == real else '✗'
    plt.title(f'{ok} pred: {pred}', fontsize=10)
plt.tight_layout(); plt.show()

## 5 · (Opcional) Fine-tuning

Podemos ir un paso más allá: **descongelar** las capas superiores de la base y reentrenarlas con un learning rate muy pequeño, para afinarlas a nuestro problema.

In [ ]:
base.trainable = True
# congelamos todo menos las últimas ~30 capas
for layer in base.layers[:-30]:
    layer.trainable = False

model.compile(optimizer=keras.optimizers.Adam(1e-5),  # LR muy pequeño
              loss='binary_crossentropy', metrics=['accuracy'])
model.fit(train_ds, validation_data=val_ds, epochs=2, verbose=2)
print('Fine-tuning terminado.')

## 6 · Tus retos

1. **Desde cero.** Entrena una CNN pequeña **sin** la base preentrenada con estos mismos (pocos) datos. ¿Qué accuracy alcanza? Compárala con el transfer learning.
2. **Otra base.** Cambia `MobileNetV2` por `ResNet50` o `EfficientNetB0`. ¿Mejora? ¿Es más lenta?
3. **Menos datos aún.** Reduce el split de entrenamiento a `train[:10%]`. El transfer learning debería aguantar bien; entrenar desde cero, no.

Cuando termines, fíjate en el paralelismo con el Tema 6: **preentrenar + fine-tuning** es la misma receta, aquí en visión y allí en texto.